# Notebook Uji Verifikasi Apify Actor (Fase 0)
## Instagram Competition Monitoring & Curation System

Notebook ini disiapkan untuk dijalankan langsung oleh pengguna/admin di Google Colab menggunakan akun Apify aktual.
Tujuan: Mengambil sampel data riil dari kedua actor untuk memverifikasi nama field output, format timestamp, dan karakteristik kuota secara pasti.

### 0. Environment Check & Smart Install (wajib dijalankan pertama)
Sel pertama mencetak versi Python dan paket, lalu memeriksa import serta signature API yang dipakai `compmon`.
Sel kedua hanya memasang paket yang **belum ada atau terlalu lama** (tidak pernah `pip install -r`, tidak menurunkan paket bawaan Colab).
Jika ada paket yang dipasang: **Runtime > Restart session**, lalu jalankan ulang notebook dari atas.


In [ ]:
# @title 0a. Environment Check
# Memeriksa Python, versi paket, import, dan signature API yang dipakai compmon.
import sys
from pathlib import Path

_root = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from compmon.env_check import run_environment_check

run_environment_check()


In [ ]:
# @title 0b. Smart Install
# Hanya memasang paket yang belum ada / di bawah versi minimum di requirements.txt.
# Tidak memakai `pip install -r` dan tidak menurunkan paket bawaan Colab.
from compmon.env_check import ensure_environment

ensure_environment()


### 1. Setup & Kredensial Apify

In [ ]:
# Import modul pihak ketiga (dependensi sudah dipasang di sel 0)
from apify_client import ApifyClient

# Inisialisasi Kredensial Apify
try:
    from google.colab import userdata
    APIFY_API_TOKEN = userdata.get('APIFY_API_TOKEN')
except Exception:
    import os
    APIFY_API_TOKEN = os.environ.get('APIFY_API_TOKEN', 'YOUR_APIFY_TOKEN_HERE')

client = ApifyClient(APIFY_API_TOKEN)
print("Client Apify siap digunakan.")


### 2. Uji 1: Actor Akun (`apify/instagram-scraper`)
Mengambil 2 postingan terbaru dari `@lombabisnis`.

In [ ]:
run_input_account = {
    "directUrls": ["https://www.instagram.com/lombabisnis/"],
    "resultsType": "posts",
    "resultsLimit": 2
}

print("Menjalankan actor apify/instagram-scraper...")
run_acc = client.actor("apify/instagram-scraper").call(run_input=run_input_account)
items_acc = client.dataset(run_acc["defaultDatasetId"]).list_items().items

print(f"Berhasil mengambil {len(items_acc)} post dari akun.")
if items_acc:
    sample = items_acc[0]
    print("Daftar Field Kunci Output:")
    for k in ['id', 'shortCode', 'url', 'timestamp', 'ownerUsername']:
        print(f"  - {k}: {sample.get(k)}")
    print("\nContoh Cuplikan Caption:", (sample.get('caption') or '')[:100], "...")

### 3. Uji 2: Actor Keyword (`seemuapps/instagram-keyword-posts-scraper`)
Mengambil 2 postingan dengan keyword `lomba business plan` (tanpa cookie).

In [ ]:
run_input_kw = {
    "keywords": ["lomba business plan"],
    "maxPostsPerKeyword": 2,
    "postType": "all"
}

print("Menjalankan actor seemuapps/instagram-keyword-posts-scraper...")
run_kw = client.actor("seemuapps/instagram-keyword-posts-scraper").call(run_input=run_input_kw)
items_kw = client.dataset(run_kw["defaultDatasetId"]).list_items().items

print(f"Berhasil mengambil {len(items_kw)} post dari keyword.")
if items_kw:
    sample_kw = items_kw[0]
    print("Daftar Field Kunci Output:")
    for k in ['postId', 'shortcode', 'url', 'takenAt', 'ownerUsername', 'keyword']:
        print(f"  - {k}: {sample_kw.get(k)}")
    print("\nContoh Cuplikan Caption:", (sample_kw.get('caption') or '')[:100], "...")

### 4. [OPT-IN] Eksperimen Biaya Pagination Akun (belum dijalankan)

Sel di bawah **hanya mengukur** dan tidak mengubah perilaku pipeline. Biaya `apify/instagram-scraper`
untuk `resultsLimit` besar **belum terverifikasi**, jadi jangan menyimpulkan apa pun sebelum sel ini
dijalankan dengan token Apify nyata dan hasilnya dicatat manual.

Syarat: `RUN_COST_EXPERIMENT = True`, satu username di `ACCOUNT_FOR_COST_TEST`, dan batas keras
`MAX_TEST_RESULTS` (default 100). Token Apify diambil dari Colab Secrets (`APIFY_API_TOKEN`).


In [ ]:
# =============================================================================
# [OPT-IN] EKSPERIMEN BIAYA PAGINATION AKUN - tidak mengubah perilaku pipeline
# Dua actor run untuk akun + cutoff yang sama: resultsLimit=20 vs MAX_TEST_RESULTS.
# Token dibaca dari Colab Secrets (APIFY_API_TOKEN); tidak ada kredensial di file ini.
# =============================================================================
RUN_COST_EXPERIMENT = False
ACCOUNT_FOR_COST_TEST = ""   # isi satu username Instagram, TANPA tanda '@'
MAX_TEST_RESULTS = 100       # batas keras jumlah hasil untuk run B

if not RUN_COST_EXPERIMENT:
    print("Dilewati. Set RUN_COST_EXPERIMENT = True untuk menjalankan eksperimen biaya.")
elif not str(ACCOUNT_FOR_COST_TEST).strip():
    print("Dilewati. Isi ACCOUNT_FOR_COST_TEST dengan satu username Instagram.")
else:
    from datetime import datetime, timedelta, timezone
    from google.colab import userdata
    from apify_client import ApifyClient

    apify = ApifyClient(userdata.get("APIFY_API_TOKEN"))
    account = str(ACCOUNT_FOR_COST_TEST).strip().lstrip("@")
    cutoff = (datetime.now(timezone.utc) - timedelta(days=60)).strftime("%Y-%m-%d")
    hard_cap = max(1, int(MAX_TEST_RESULTS))

    def _shown(value):
        return "unknown" if value is None else value

    def run_once(label, limit):
        run_input = {
            "directUrls": ["https://www.instagram.com/" + account + "/"],
            "resultsType": "posts",
            "resultsLimit": int(limit),
            "onlyPostsNewerThan": cutoff,
        }
        run = apify.actor("apify/instagram-scraper").call(run_input=run_input)
        items = apify.dataset(run["defaultDatasetId"]).list_items().items or []
        in_window, older = 0, 0
        for item in items:
            stamp = str(item.get("timestamp") or "")[:10]
            if stamp and stamp < cutoff:
                older += 1
            else:
                in_window += 1
        total_usd = run.get("usageTotalUsd")
        usage = run.get("usage")
        has_cost = isinstance(total_usd, (int, float))
        per_item = (total_usd / len(items)) if has_cost and items else None
        print("")
        print("=== " + label + ": resultsLimit=" + str(limit) + " ===")
        print("onlyPostsNewerThan          : " + cutoff)
        print("items returned (raw)        : " + str(len(items)))
        print("items inside the window     : " + str(in_window))
        print("items older than the cutoff : " + str(older))
        print("usageTotalUsd               : " + str(_shown(total_usd)))
        print("usage breakdown             : " + str(_shown(usage)))
        print("cost per returned item      : " + str(_shown(per_item)))
        return {"limit": int(limit), "items": len(items), "in_window": in_window,
                "older": older, "total_usd": total_usd if has_cost else None}

    small = run_once("RUN A", min(20, hard_cap))
    big = run_once("RUN B", hard_cap)

    print("")
    print("=== KESIMPULAN (untuk pemilik proyek) ===")
    if big["older"] == 0:
        print("Run B mengembalikan HANYA post di dalam window: ya")
    else:
        print("Run B mengembalikan HANYA post di dalam window: tidak ("
              + str(big["older"]) + " post lebih tua dari cutoff)")
    if small["total_usd"] is None or big["total_usd"] is None:
        print("Biaya billed sesuai jumlah item: unknown (usageTotalUsd tidak tersedia)")
    elif not small["items"] or not big["items"]:
        print("Biaya billed sesuai jumlah item: unknown (tidak ada item yang dikembalikan)")
    else:
        a_per = small["total_usd"] / small["items"]
        b_per = big["total_usd"] / big["items"]
        ratio = (b_per / a_per) if a_per else None
        print("biaya per item: run A=" + str(round(a_per, 6)) + " USD, run B="
              + str(round(b_per, 6)) + " USD, rasio=" + str(_shown(round(ratio, 3) if ratio else None)))
        close = ratio is not None and 0.75 <= ratio <= 1.25
        print("Biaya billed sesuai jumlah item: " + ("ya" if close else "tidak"))
    print("")
    print("Catat output di atas secara manual; docs tetap 'Not tested (needs real services)'"
          " sebelum ada output nyata.")
